# Structure Abnormal Detection CNN v1.0

- 목적: `major 4 structure` vs `others(abnormal)` 이진 분류
- 라벨 소스: `data/train_structure.csv`, `data/dev_structure.csv`
- 이미지 입력: `front.png`, `top.png`


In [ ]:
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.metrics import roc_auc_score

ROOT = (Path.cwd() / '../../').resolve()
DATA_DIR = ROOT / 'data'
OUT_DIR = ROOT / 'outputs' / 'structure_abnormal_detection_cnn_v1.0'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'SEED': 42,
    'IMG_SIZE': 224,
    'BACKBONE_NAME': 'convnext_tiny.fb_in22k_ft_in1k',
    'BATCH_SIZE': 32,
    'NUM_WORKERS': 8,
    'EPOCHS': 25,
    'LEARNING_RATE': 2e-4,
    'WEIGHT_DECAY': 1e-4,
    'MIN_LR': 1e-6,
    'EARLY_STOPPING_PATIENCE': 5,
}

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('ROOT:', ROOT)


In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')

train_struct = pd.read_csv(DATA_DIR / 'train_structure.csv', encoding='utf-8-sig')
dev_struct = pd.read_csv(DATA_DIR / 'dev_structure.csv', encoding='utf-8-sig')

train_df = train_df.merge(train_struct[['id', 'structure_index']], on='id', how='left')
dev_df = dev_df.merge(dev_struct[['id', 'structure_index']], on='id', how='left')

assert train_df['structure_index'].notna().all(), 'train 구조 라벨 누락'
assert dev_df['structure_index'].notna().all(), 'dev 구조 라벨 누락'

major4 = (
    train_df['structure_index']
    .value_counts()
    .head(4)
    .index
    .astype(int)
    .tolist()
)
major4_set = set(major4)

# abnormal=1: major4 외의 구조
train_df['abnormal_label'] = (~train_df['structure_index'].astype(int).isin(major4_set)).astype(np.float32)
dev_df['abnormal_label'] = (~dev_df['structure_index'].astype(int).isin(major4_set)).astype(np.float32)

print('major4 indices:', major4)
print('train abnormal ratio:', float(train_df['abnormal_label'].mean()))
print('dev abnormal ratio:', float(dev_df['abnormal_label'].mean()))


In [ ]:
class TwoViewDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, transform, is_test: bool = False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        sample_id = str(row['id'])
        sample_dir = self.root_dir / sample_id

        front = Image.open(sample_dir / 'front.png').convert('RGB')
        top = Image.open(sample_dir / 'top.png').convert('RGB')
        front = self.transform(front)
        top = self.transform(top)

        if self.is_test:
            return {'id': sample_id, 'front': front, 'top': top}

        label = float(row['abnormal_label'])
        return {'id': sample_id, 'front': front, 'top': top, 'label': torch.tensor(label, dtype=torch.float32)}


train_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((CFG['IMG_SIZE'], CFG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = TwoViewDataset(train_df, DATA_DIR / 'train', train_transform, is_test=False)
dev_dataset = TwoViewDataset(dev_df, DATA_DIR / 'dev', test_transform, is_test=False)
test_dataset = TwoViewDataset(test_df, DATA_DIR / 'test', test_transform, is_test=True)

loader_kwargs = dict(batch_size=CFG['BATCH_SIZE'], num_workers=CFG['NUM_WORKERS'], pin_memory=(device.type == 'cuda'))
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
dev_loader = DataLoader(dev_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print('train/dev/test:', len(train_dataset), len(dev_dataset), len(test_dataset))


In [ ]:
class TwoViewAbnormalCNN(nn.Module):
    def __init__(self, backbone_name: str):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1),
        )

    def forward(self, front: torch.Tensor, top: torch.Tensor) -> torch.Tensor:
        f_front = self.backbone(front)
        f_top = self.backbone(top)
        feat = torch.cat([f_front, f_top], dim=1)
        return self.head(feat).view(-1)

model = TwoViewAbnormalCNN(CFG['BACKBONE_NAME']).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR'])
criterion = nn.BCEWithLogitsLoss()


In [ ]:
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    all_logits = []
    all_labels = []

    for batch in tqdm(loader, desc='Eval', dynamic_ncols=True, ascii=True):
        front = batch['front'].to(device)
        top = batch['top'].to(device)
        labels = batch['label'].to(device)

        logits = model(front, top)
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())

    logits = torch.cat(all_logits).numpy().astype(np.float64)
    labels = torch.cat(all_labels).numpy().astype(np.float64)
    probs = 1.0 / (1.0 + np.exp(-logits))
    p = np.clip(probs, 1e-15, 1 - 1e-15)

    logloss = float(-np.mean(labels * np.log(p) + (1.0 - labels) * np.log(1.0 - p)))
    acc = float(np.mean((probs > 0.5) == labels))
    auc = float(roc_auc_score(labels, probs))
    return {'dev_logloss': logloss, 'dev_acc': acc, 'dev_auc': auc}, probs, labels


def train_one_epoch(model: nn.Module, loader: DataLoader):
    model.train()
    rows = []
    for batch in tqdm(loader, desc='Train', dynamic_ncols=True, ascii=True):
        front = batch['front'].to(device)
        top = batch['top'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(front, top)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        rows.append({'loss': float(loss.item())})

    return float(np.mean([x['loss'] for x in rows]))


best_logloss = float('inf')
patience_left = CFG['EARLY_STOPPING_PATIENCE']
history = []
best_path = OUT_DIR / 'best_model.pt'

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_loss = train_one_epoch(model, train_loader)
    dev_metrics, _, _ = evaluate(model, dev_loader)
    scheduler.step()

    improved = dev_metrics['dev_logloss'] < best_logloss
    if improved:
        best_logloss = dev_metrics['dev_logloss']
        patience_left = CFG['EARLY_STOPPING_PATIENCE']
        torch.save({'model_state_dict': model.state_dict(), 'cfg': CFG, **dev_metrics}, best_path)
    else:
        patience_left -= 1

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'lr': float(optimizer.param_groups[0]['lr']),
        **dev_metrics,
        'improved': bool(improved),
        'patience_left': int(patience_left),
    }
    history.append(row)
    print(row)

    if patience_left < 0:
        print('early stop at epoch', epoch)
        break

history_df = pd.DataFrame(history)
history_path = OUT_DIR / 'history.csv'
history_df.to_csv(history_path, index=False)
print('saved history:', history_path)
print('best_logloss:', best_logloss)


In [ ]:
ckpt = torch.load(OUT_DIR / 'best_model.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])

# dev predictions
dev_metrics, dev_probs, dev_labels = evaluate(model, dev_loader)
print('best dev metrics:', dev_metrics)

dev_pred_df = dev_df[['id', 'structure_index', 'abnormal_label']].copy()
dev_pred_df['abnormal_prob'] = dev_probs
dev_pred_df['major4_prob'] = 1.0 - dev_pred_df['abnormal_prob']
dev_pred_path = OUT_DIR / 'dev_predictions.csv'
dev_pred_df.to_csv(dev_pred_path, index=False, encoding='utf-8-sig')
print('saved:', dev_pred_path)

# test predictions
@torch.no_grad()
def predict_test(model: nn.Module, loader: DataLoader):
    model.eval()
    ids = []
    probs = []
    for batch in tqdm(loader, desc='Test', dynamic_ncols=True, ascii=True):
        front = batch['front'].to(device)
        top = batch['top'].to(device)
        logits = model(front, top)
        p = torch.sigmoid(logits).cpu().numpy().astype(np.float64)
        ids.extend(batch['id'])
        probs.extend(p.tolist())
    return ids, np.array(probs, dtype=np.float64)

ids, test_probs = predict_test(model, test_loader)
test_pred_df = pd.DataFrame({'id': ids, 'abnormal_prob': test_probs, 'major4_prob': 1.0 - test_probs})
test_pred_path = OUT_DIR / 'test_predictions.csv'
test_pred_df.to_csv(test_pred_path, index=False, encoding='utf-8-sig')
print('saved:', test_pred_path)

# optional submission-like file
submission_path = ROOT / 'outputs' / 'submissions' / 'structure_abnormal_detection_cnn_v1.0.csv'
submission_path.parent.mkdir(parents=True, exist_ok=True)
test_pred_df.to_csv(submission_path, index=False, encoding='utf-8-sig')
print('saved:', submission_path)
